# Conversational Memory in LangChain

## Table of Contents
1. [Without Memory — The Problem](#without-memory)
2. [With Memory — The Benefits](#with-memory)
3. [Types of Memory](#types-of-memory)
   - 3a. ConversationBufferMemory
   - 3b. ConversationBufferWindowMemory
   - 3c. ConversationSummaryMemory
   - 3d. ConversationSummaryBufferMemory
   - 3e. ConversationTokenBufferMemory
4. [Which Memory to Choose?](#which-memory)
5. [Best Practices](#best-practices)

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from dotenv import load_dotenv

load_dotenv(override=True)

llm = ChatAnthropic(model='claude-sonnet-4-6', max_tokens=300)

---
## 1. Without Memory — The Problem

By default, every call to an LLM is **stateless** — it remembers nothing from previous turns.

```
Turn 1:  You  --> "My name is Ali"
         Bot  --> "Nice to meet you, Ali!"

Turn 2:  You  --> "What is my name?"
         Bot  --> "I don't know your name."   ← PROBLEM!
```

Each `.invoke()` is a **fresh, isolated API call** — no history is carried over.

In [ ]:
# ❌ WITHOUT MEMORY — bot forgets everything

chain = llm | StrOutputParser()

# Turn 1
print("User: My name is Ali and I love Python.")
r1 = chain.invoke('My name is Ali and I love Python.')
print('AI Response:', r1)
print()

# Turn 2 — bot has NO idea who we are
print("User: What is my name and what do I love?")
r2 = chain.invoke('What is my name and what do I love?')
print('AI Response:', r2)   # Will say: "I don't know your name"

---
## 2. With Memory — The Benefits

Conversational memory **injects past messages** into every new prompt, so the model has context.

| Without Memory | With Memory |
|---|---|
| Every turn is isolated | Full conversation history available |
| Cannot answer follow-up questions | Handles "what did I just say?" |
| No personalization | Remembers user name, preferences |
| Feels like talking to a stranger every time | Feels like a real conversation |

> **Analogy:** Without memory = goldfish brain (forgets every 3 seconds).  
> With memory = normal human brain (remembers what was just said).

---
## 3. Types of Memory

| Memory Type | How It Works | Best For |
|---|---|---|
| **BufferMemory** | Stores ALL messages | Short chats |
| **BufferWindowMemory** | Stores last K messages only | Medium chats |
| **SummaryMemory** | Summarizes history with LLM | Very long chats |
| **SummaryBufferMemory** | Summary + recent messages | Long chats with detail |
| **TokenBufferMemory** | Limits history by token count | Token-budget control |

---
### 3a. ConversationBufferMemory

**Stores every single message** — the simplest form of memory.

```
History stored:
  Human: My name is Ali
  AI:    Nice to meet you, Ali!
  Human: I love Python
  AI:    Great choice!
  Human: What is my name?          <-- new message
```

**Pro:** Never forgets anything.  
**Con:** History grows forever → expensive token usage for long chats.

In [ ]:
# ✅ ConversationBufferMemory (Legacy API) — simplest way to add memory

from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

''' Fixed. langchain.memory and langchain.chains were fully removed in LangChain 1.x. The cell now uses InMemoryChatMessageHistory from langchain_core — the direct replacement — with the same pattern:

memory.add_user_message() / memory.add_ai_message() mirrors the old ConversationBufferMemory API
memory.messages gives the full stored history (same as the old memory.chat_memory.messages)
The chat() helper keeps the same simple call style as the old conversation.predict()'''

# memory stores the full chat history automatically
memory = ConversationBufferMemory(return_messages=True)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False,   # set True to see the full prompt sent to the LLM
)

# Turn 1
r1 = conversation.predict(input='My name is Ali and I love Python.')
print('Turn 1:', r1)
print()

# Turn 2 — memory carries over automatically
r2 = conversation.predict(input='What is my name?')
print('Turn 2:', r2)   # Correctly: Ali
print()

# Turn 3
r3 = conversation.predict(input='What programming language do I love?')
print('Turn 3:', r3)   # Correctly: Python
print()

# Inspect what is stored in memory
print('--- Memory buffer (string) ---')
print(memory.buffer)
print()
print('--- Memory messages (objects) ---')
for msg in memory.chat_memory.messages:
    print(f'{msg.__class__.__name__}: {msg.content}')

In [ ]:
# ✅ ConversationBufferMemory — stores ALL messages manually
# Modern LangChain (LCEL) manages memory by passing chat_history directly

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Prompt with a placeholder for history
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Be concise.'),
    MessagesPlaceholder(variable_name='chat_history'),   # ← history injected here
    ('human', '{input}'),
])

chain = prompt | llm | StrOutputParser()

# In-memory store — one history object per session_id
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Wrap chain with message history (Buffer — keeps ALL messages)
chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='chat_history',
)

config = {'configurable': {'session_id': 'session_1'}}

# Turn 1
r1 = chain_with_memory.invoke({'input': 'My name is Ali and I love Python.'}, config=config)
print('Turn 1:', r1)
print()

# Turn 2
r2 = chain_with_memory.invoke({'input': 'What is my name?'}, config=config)
print('Turn 2:', r2)   # Now correctly answers: Ali!
print()

# Turn 3
r3 = chain_with_memory.invoke({'input': 'What programming language do I love?'}, config=config)
print('Turn 3:', r3)   # Correctly answers: Python!

# Inspect full stored history
print('\n--- Full History ---')
for msg in store['session_1'].messages:
    print(f'{msg.__class__.__name__}: {msg.content}')

---
### 3b. ConversationBufferWindowMemory

**Stores only the last K turns** — older messages are discarded.

```
K = 2  (keep last 2 turns)

Turn 1: Human: Hi, I'm Ali          ← will be forgotten after turn 3
Turn 2: Human: I love Python        ← will be forgotten after turn 4
Turn 3: Human: What's 2+2?          ← kept
Turn 4: Human: What is my name?     ← kept  (Ali is now forgotten!)
```

**Pro:** Bounded memory — token cost stays constant.  
**Con:** Forgets older context.

In [ ]:
# ✅ Window Memory — keep only last K messages
# We implement this by trimming the history before passing it

from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnableLambda
from operator import itemgetter

WINDOW_SIZE = 4   # keep last 4 messages (2 human + 2 AI)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Be concise.'),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human', '{input}'),
])

# Trim history to last WINDOW_SIZE messages before passing to prompt
def trim_history(inputs):
    inputs['chat_history'] = inputs['chat_history'][-WINDOW_SIZE:]
    return inputs

windowed_chain = RunnableLambda(trim_history) | prompt | llm | StrOutputParser()

store_w = {}

def get_session_history_w(session_id: str):
    if session_id not in store_w:
        store_w[session_id] = InMemoryChatMessageHistory()
    return store_w[session_id]

windowed_chain_with_memory = RunnableWithMessageHistory(
    windowed_chain,
    get_session_history_w,
    input_messages_key='input',
    history_messages_key='chat_history',
)

config_w = {'configurable': {'session_id': 'window_session'}}

r1 = windowed_chain_with_memory.invoke({'input': 'My name is Ali.'}, config=config_w)
print('Turn 1:', r1)

r2 = windowed_chain_with_memory.invoke({'input': 'I am from India.'}, config=config_w)
print('Turn 2:', r2)

r3 = windowed_chain_with_memory.invoke({'input': 'I love Python.'}, config=config_w)
print('Turn 3:', r3)

# Turn 4: Window only shows last 4 messages — earliest context may be lost
r4 = windowed_chain_with_memory.invoke({'input': 'What is my name and where am I from?'}, config=config_w)
print('Turn 4:', r4)   # May forget 'Ali' depending on window size

---
### 3c. ConversationSummaryMemory

**Uses the LLM to summarize the conversation** instead of storing raw messages.

```
Raw history:
  Human: My name is Ali
  AI:    Nice to meet you, Ali!
  Human: I live in Kolkata
  AI:    Great city!
  Human: I work as a data scientist
  AI:    Interesting!

Summary stored:
  "The user's name is Ali, lives in Kolkata, and works as a data scientist."
```

**Pro:** Memory stays compact no matter how long the chat.  
**Con:** Fine details may be lost in the summary. Costs extra LLM calls to summarize.

In [ ]:
# ✅ Summary Memory — manually maintain a running summary

summary_llm = ChatAnthropic(model='claude-sonnet-4-6', max_tokens=150)

def summarize_history(history: list, new_human: str, new_ai: str) -> str:
    """Ask the LLM to produce an updated summary."""
    if not history:
        existing_summary = 'No previous conversation.'
    else:
        existing_summary = history[0].content   # we store summary as a single SystemMessage

    summarize_prompt = f"""Summarize the conversation so far in 1-2 sentences.

Current summary: {existing_summary}

New exchange:
Human: {new_human}
AI: {new_ai}

Updated summary:"""
    return summary_llm.invoke(summarize_prompt).content


# Chat loop with running summary
from langchain_core.messages import SystemMessage

summary_history = []   # stores a single SystemMessage with the current summary

chat_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Be concise. Here is context: {summary}'),
    ('human', '{input}'),
])
chat_chain = chat_prompt | llm | StrOutputParser()

turns = [
    'My name is Ali and I live in Kolkata.',
    'I work as a data scientist.',
    'What do you know about me so far?',   # should recall name, city, job
]

current_summary = ''
for human_msg in turns:
    ai_response = chat_chain.invoke({'summary': current_summary, 'input': human_msg})
    print(f'Human: {human_msg}')
    print(f'AI:    {ai_response}')
    print()

    # Update the running summary after each exchange
    current_summary = summarize_history([], human_msg, ai_response) if not current_summary \
                      else summarize_history([SystemMessage(content=current_summary)], human_msg, ai_response)

print('--- Final Summary Stored ---')
print(current_summary)

---
### 3d. ConversationSummaryBufferMemory

**Hybrid** — keeps recent messages in full AND maintains a summary of older messages.

```
What's stored:

  [Summary of old turns]              ← compressed older context
  Human: What's my favourite food?    ← recent raw message
  AI:    I don't know yet.            ← recent raw message
  Human: It's biryani!                ← recent raw message  (current turn)
```

**Pro:** Best of both worlds — recent details preserved, old context summarized.  
**Con:** Slightly more complex to manage.

In [ ]:
# ✅ SummaryBuffer Memory — summary of old + raw recent messages

RECENT_WINDOW = 4   # keep last 4 messages raw; summarize everything older

class SummaryBufferChat:
    def __init__(self, llm, window=4):
        self.llm = llm
        self.window = window
        self.summary = ''           # running summary of old messages
        self.recent_messages = []   # last N raw messages

    def _update_summary(self, old_messages):
        """Summarize messages that are being pushed out of the window."""
        history_text = '\n'.join(
            f"{m.__class__.__name__}: {m.content}" for m in old_messages
        )
        p = f"Summarize this conversation in 1-2 sentences:\n{history_text}"
        new_summary = self.llm.invoke(p).content
        if self.summary:
            self.summary = f"{self.summary} {new_summary}"
        else:
            self.summary = new_summary

    def chat(self, user_input: str) -> str:
        # Build prompt with summary + recent messages
        system_content = 'You are a helpful assistant. Be concise.'
        if self.summary:
            system_content += f'\n\nContext from earlier: {self.summary}'

        messages = [('system', system_content)]
        for m in self.recent_messages:
            role = 'human' if isinstance(m, HumanMessage) else 'assistant'
            messages.append((role, m.content))
        messages.append(('human', user_input))

        prompt = ChatPromptTemplate.from_messages(messages)
        response = (prompt | self.llm | StrOutputParser()).invoke({})

        # Add new exchange to recent messages
        self.recent_messages.append(HumanMessage(content=user_input))
        self.recent_messages.append(AIMessage(content=response))

        # If window exceeded, summarize oldest messages
        if len(self.recent_messages) > self.window:
            overflow = self.recent_messages[:-self.window]
            self._update_summary(overflow)
            self.recent_messages = self.recent_messages[-self.window:]

        return response


bot = SummaryBufferChat(llm, window=4)

print(bot.chat('My name is Ali and I love biryani.'))
print()
print(bot.chat('I am a Python developer.'))
print()
print(bot.chat('I have 5 years of experience.'))
print()
print(bot.chat('What do you know about me?'))   # uses summary + recent messages
print()
print('--- Summary stored ---')
print(bot.summary)
print('--- Recent messages ---')
for m in bot.recent_messages:
    print(f'{m.__class__.__name__}: {m.content}')

---
### 3e. ConversationTokenBufferMemory

**Limits history by token count**, not by number of messages.

```
Max tokens = 200

Message history (tokens):
  Human: Tell me about Python  (8 tokens)   ← dropped (too old, over limit)
  AI:    Python is a language... (40 tokens) ← dropped
  Human: What about JavaScript? (6 tokens)  ← kept
  AI:    JS runs in browsers... (35 tokens)  ← kept
  Human: Compare both languages (5 tokens)  ← current turn
  Total kept: ~86 tokens  ✓ under 200
```

**Pro:** Precise token budget control — no surprises on LLM costs.  
**Con:** Requires counting tokens (needs a tokenizer).

In [ ]:
# ✅ Token Buffer Memory — trim history by token count

from langchain_core.messages import trim_messages
from langchain_anthropic import ChatAnthropic

MAX_TOKENS = 300   # maximum tokens allowed in history

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Be concise.'),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human', '{input}'),
])

# trim_messages trims from oldest first to stay within token budget
trimmer = trim_messages(
    max_tokens=MAX_TOKENS,
    strategy='last',          # keep the LAST (most recent) messages
    token_counter=llm,        # use Claude to count tokens
    include_system=True,      # always keep the system message
    allow_partial=False,      # never cut a message in half
    start_on='human',         # always start history on a human turn
)

from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

# Trim history before passing into prompt
token_chain = (
    RunnablePassthrough.assign(chat_history=itemgetter('chat_history') | trimmer)
    | prompt
    | llm
    | StrOutputParser()
)

store_t = {}

def get_session_history_t(session_id: str):
    if session_id not in store_t:
        store_t[session_id] = InMemoryChatMessageHistory()
    return store_t[session_id]

token_chain_with_memory = RunnableWithMessageHistory(
    token_chain,
    get_session_history_t,
    input_messages_key='input',
    history_messages_key='chat_history',
)

config_t = {'configurable': {'session_id': 'token_session'}}

r1 = token_chain_with_memory.invoke({'input': 'My name is Ali.'}, config=config_t)
print('Turn 1:', r1)

r2 = token_chain_with_memory.invoke({'input': 'I love Python and data science.'}, config=config_t)
print('Turn 2:', r2)

r3 = token_chain_with_memory.invoke({'input': 'What do you remember about me?'}, config=config_t)
print('Turn 3:', r3)

---
## 4. Which Memory to Choose?

```
Chat length?  Short (< 10 turns)  --> BufferMemory          (simplest)
              Medium              --> BufferWindowMemory     (fixed cost)
              Long (100s of turns)--> SummaryMemory          (compact)
              Long + need detail  --> SummaryBufferMemory    (best of both)
              Need token control  --> TokenBufferMemory      (budget aware)
```

| Memory | Stores | Token Cost | Remembers Details |
|---|---|---|---|
| Buffer | All messages | Grows forever | Yes — everything |
| BufferWindow | Last K messages | Fixed | Only recent |
| Summary | LLM summary | Small, fixed | Gist only |
| SummaryBuffer | Summary + recent raw | Medium | Recent details + old gist |
| TokenBuffer | Messages up to N tokens | Controlled | Recent (by tokens) |

---
## 5. Best Practices

| Practice | Why |
|---|---|
| Use `session_id` per user | Isolate memory across different users |
| Don't use BufferMemory for production | Token cost grows unbounded |
| Prefer SummaryBuffer for chatbots | Good balance of detail and cost |
| Always inject history via `MessagesPlaceholder` | Clean, structured history passing |
| Test memory with `.messages` inspection | Verify what's actually stored |

---
## Summary

```
No Memory         →  Every call is isolated. Forgets everything.
BufferMemory      →  Remember all. Best for short chats.
WindowMemory      →  Remember last K. Bounded cost.
SummaryMemory     →  Summarize history. Compact. Loses detail.
SummaryBuffer     →  Summary of old + recent raw. Best balance.
TokenBuffer       →  Keep messages within token budget.
```

> **Golden rule:** Always pass history as messages through `MessagesPlaceholder`.  
> The difference between memory types is *how much* history you pass, not *how* you pass it.